### "Statistics and Data Analysis" 24: The Chi-squared Test: Are Two Distributions the Same?
In this lecture, Dr. Brunton asks a fundamental question in statistics: are two data sets from the same distribution or different distributions?  We answer this using the Chi-squared test, with a python example.  

https://www.youtube.com/watch?v=63S3FLISKMs&list=PLMrJAkhIeNNT14qn1c5qdL29A1UaHamjx&index=24

In [ ]:
# Alpha particles emmitted by Americium 241
# Example from Rice, taken from Berkson 1966
# 10220 Gieger clikcs (emissions) were measured, 
# specifically time between clicks data is binned into 10-second intervals
# Clicks per 10-second interval should be Poisson (time between is exponential)

import numpy as np
import matplotlib.pyplot as plt

# 1. Raw Data from Americium 241 experiment (counts in 10s intervals)
click_tenseconds = np.arange(0, 18) # Number of counts per 10-second interval
observed_counts = np.array([1, 6, 11, 28, 56, 105, 126, 146, 164, 161, 123, 101, 74, 53, 23, 15, 9, 5])

# 2. Plotting
plt.figure(figsize=(10, 6))

# Plotting the observed data
plt.plot(click_tenseconds, observed_counts,  'o-',color='blue', linewidth=2, label='Observed Data (Am-241)')

# Adding labels and title
plt.title('Observed Alpha Particle Emission per 10-Second Interval', fontsize=14)
plt.xlabel('Number of Click per 10s Interval', fontsize=12)
plt.ylabel('Observed Count', fontsize=12)
plt.legend()
plt.grid(True)
plt.xticks(click_tenseconds)
plt.show()

In [ ]:
from scipy.stats import poisson

# Calculate the sample mean
sample_mean = np.average(click_tenseconds, weights=observed_counts)

# total number of observations
total_count = np.sum(observed_counts)

# Fit a Poisson distribution using the sample mean as lambda
lambda_hat = sample_mean
poisson_dist = poisson.pmf(click_tenseconds, mu=lambda_hat)* total_count

# Plotting
plt.figure(figsize=(10, 6))
# Plotting the observed data
plt.plot(click_tenseconds, observed_counts, 'o-', linewidth=2, color='blue', label='Observed Data (Am-241)')

# Plot the theoretical fit as a line with markers
plt.plot(click_tenseconds, poisson_dist, 'x--', color='red', linewidth=2, label=f'Poisson Fit ($\\hat{{\\lambda}}$={lambda_hat:.2f})')

# Adding labels and title
plt.title('Fitting a Poisson Distribution to Radioactive Decay Data', fontsize=14)
plt.xlabel('Number of Click per 10s Interval', fontsize=12)
plt.ylabel('Observed Count', fontsize=12)
plt.legend()
plt.grid(True)

plt.xticks(click_tenseconds)
plt.show()

In [ ]:
from scipy.stats import chi2

# 1. Raw Data from Americium 241 experiment (counts in 10s intervals)
click_tenseconds = np.arange(0, 18) # Number of counts per 10-second interval
observed_counts = np.array([1, 6, 11, 28, 56, 105, 126, 146, 164, 161, 123, 101, 74, 53, 23, 15, 9, 5])

# Combine the first three bins (0, 1, and 2 counts into one bin)
observed_combined = np.copy(observed_counts)
observed_combined[2] = np.sum(observed_combined[0:3]) # Sum of the first three bins
observed_combined = np.delete(observed_combined, [0, 1]) # remove the first two bins

# Adjust click_tenseconds to match the combined bins
click_combined = np.copy(click_tenseconds)
click_combined = np.delete(click_combined, [0, 1])

# Calculate the sample mean
sample_mean = np.average(click_tenseconds, weights=observed_counts)

# total number of observations
total_count = np.sum(observed_counts)

# Compute expected frequencies for each bin using Poisson distribution
lambda_hat = sample_mean
expected = poisson.pmf(click_tenseconds, mu=lambda_hat)* total_count

# Combine the first three bins of expected frequencies as well
expected_combined = np.copy(expected)
expected_combined[2] = np.sum(expected_combined[0:3]) # Sum of the first three bins
expected_combined = np.delete(expected_combined, [0, 1]) # remove the first two bins

# Perform the chi-squared test
chi_squared_stat = np.sum((observed_combined - expected_combined)**2/expected_combined)

# Degrees of freedom (number of bins - 1 for lambda estimate - 1)
degrees_of_freedom = len(click_combined) - 1 - 1 

# Calculte the p-value
p_value = 1 - chi2.cdf(chi_squared_stat, df=degrees_of_freedom)

print(f'Chi-squared statistic: {chi_squared_stat:.4f}')
print(f'Degrees of freedom: {degrees_of_freedom}')
print(f'P-value: {p_value:.4f}')

# Plotting the chi-squared PDF
x = np.linspace(0, 40, 1000)
pdf = chi2.pdf(x, df=degrees_of_freedom)

# Rejection region for 95% confidence interval (alpha = 0.05)
critical_value = chi2.ppf(1 - 0.05, df=degrees_of_freedom)

plt.figure(figsize=(10, 6))

# Plotting the chi-squared PDF
plt.plot(x, pdf, 'b-', linewidth=2, label=f'Chi-squared PDF (df={degrees_of_freedom})')

# Highlight the rejection region
plt.fill_between(x, pdf, where=(x >= critical_value), color='red', alpha=0.3, label='Rejection Region')

# Add the Chi-Squared statistic as a vertical line
plt.axvline(x=chi_squared_stat, color='green', linestyle='--', label=f'Chi-squared statistic: {chi_squared_stat:.4f}')

# Add the critical value as a vertical line
plt.axvline(x=critical_value, color='red', linestyle='--', label=f'Critical value: {critical_value:.4f}')

# Add labels, title, legend, and grid
plt.title('Chi-squared Test Probability Density Function', fontsize=14)
plt.xlabel('Chi-squared Value', fontsize=12)
plt.ylabel('Probability Density', fontsize=12)
plt.legend()
plt.grid(True)

plt.show()

# Print rejection criterion
if chi_squared_stat < critical_value:
    print("Fail to reject the null hypothesis: The data is consistent with the Poisson distribution.")
else:
    print("Reject the null hypothesis: The data does not fit the Poisson distribution.")
